# Layer 5 Graph Visualization Checkpoint

This notebook visualizes dependency risk propagation with:
1. Raw graph view
2. Risk-colored graph view
3. Suspicious path highlighting and interpretation

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

from detector.layer5_graph.graph_builder import build_dependency_graph
from detector.layer5_graph.graph_analyzer import propagate_risk

PACKAGE = "requests"
REGISTRY = "pypi"

graph = build_dependency_graph(PACKAGE, REGISTRY, max_depth=2)
scores = propagate_risk(graph, {PACKAGE: 75})

print(f"nodes={graph.number_of_nodes()} edges={graph.number_of_edges()}")

In [ ]:
plt.figure(figsize=(12, 6))
pos = nx.spring_layout(graph, seed=42)
nx.draw_networkx(
    graph,
    pos=pos,
    with_labels=True,
    node_size=650,
    font_size=8,
    arrows=True,
    edge_color="#888888"
)
plt.title("Raw Dependency Graph")
plt.axis("off")
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
pos = nx.spring_layout(graph, seed=42)
risk_values = [scores.get(node, {}).get("final_score", 0.0) for node in graph.nodes]
nodes = nx.draw_networkx_nodes(
    graph,
    pos=pos,
    node_color=risk_values,
    cmap=plt.cm.Reds,
    node_size=700
)
nx.draw_networkx_edges(graph, pos=pos, arrows=True, edge_color="#555555")
nx.draw_networkx_labels(graph, pos=pos, font_size=8)
plt.colorbar(nodes, label="Risk score")
plt.title("Risk-Colored Dependency Graph")
plt.axis("off")
plt.show()

In [ ]:
suspicious_path = []
for node, detail in scores.items():
    if detail.get("final_score", 0) >= 70:
        try:
            suspicious_path = nx.shortest_path(graph, source=PACKAGE, target=node)
            break
        except Exception:
            continue

print("Highlighted suspicious path:", suspicious_path if suspicious_path else "None")

if suspicious_path:
    edges = list(zip(suspicious_path[:-1], suspicious_path[1:]))
    plt.figure(figsize=(12, 6))
    pos = nx.spring_layout(graph, seed=42)
    nx.draw_networkx(graph, pos=pos, with_labels=True, node_size=650, font_size=8, edge_color="#BBBBBB")
    nx.draw_networkx_edges(graph, pos=pos, edgelist=edges, width=3, edge_color="#D62728")
    plt.title("Highlighted Suspicious Path")
    plt.axis("off")
    plt.show()

### Interpretation

Risk propagation reflects how suspiciousness can travel across direct and transitive dependencies.
Nodes closer to high-risk sources tend to receive stronger propagated scores due to distance-based decay.
The highlighted path illustrates one potentially concerning chain that should be reviewed first during triage.